# Experiment: Compare Weather Models

Objective:
- load completed baseline artifacts and GNN checkpoints
- evaluate all models on the same monthly target set
- compute standard downscaling metrics
- generate compact comparison plots

Metrics included:
- RMSE
- MAE
- bias
- correlation
- R2
- NSE
- KGE
- precipitation event metrics: POD, FAR, CSI, frequency bias

In [ ]:
from __future__ import annotations

import json
import math
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from torch.utils.data import DataLoader

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
with (ROOT / 'configs' / 'default.yaml').open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

import sys
sys.path.insert(0, str(ROOT))

from src.data.dataset import WeatherGraphDataset
from src.models import NodewiseMLPBaseline, build_mlp_baseline_from_config, build_pignn_from_config
from src.training.train import collate_samples

TARGET_NAMES = ['temperature', 'precipitation', 'u_wind', 'v_wind']
EVENT_THRESHOLD = 0.1


## Evaluation Setup

Point this at the months you want to compare. If left as `None`, the notebook uses every month that has both dynamic inputs and targets.

In [ ]:
MONTHS = None
DEVICE = torch.device(CONFIG['training']['device'])

graph_path = ROOT / CONFIG['paths']['graph_output']
dynamic_dir = ROOT / CONFIG['era5']['processed_output_dir']
target_dir = ROOT / CONFIG['targets']['output_dir']


def month_from_path(path: Path) -> str:
    return path.stem.rsplit('_', 1)[-1]


dynamic = {month_from_path(path): path for path in sorted(dynamic_dir.glob('era5_dynamic_*.pt'))}
targets = {month_from_path(path): path for path in sorted(target_dir.glob('targets_*.pt'))}
months = sorted(set(dynamic) & set(targets))
if MONTHS:
    months = [month for month in months if month in set(MONTHS)]

assert months, 'No paired months found.'
print('months', months[:5], '...', months[-5:])
print('count', len(months))


In [ ]:
dataset = WeatherGraphDataset(
    graph_path,
    [dynamic[m] for m in months],
    [targets[m] for m in months],
)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_samples)
print('timesteps', len(dataset))


## Metrics

These are implemented directly in the notebook so the comparison stays self-contained.

In [ ]:
def _finite_pair(y_true, y_pred):
    mask = torch.isfinite(y_true) & torch.isfinite(y_pred)
    return y_true[mask], y_pred[mask]


def rmse(y_true, y_pred):
    y_true, y_pred = _finite_pair(y_true, y_pred)
    return torch.sqrt(((y_pred - y_true) ** 2).mean()).item()


def mae(y_true, y_pred):
    y_true, y_pred = _finite_pair(y_true, y_pred)
    return (y_pred - y_true).abs().mean().item()


def bias(y_true, y_pred):
    y_true, y_pred = _finite_pair(y_true, y_pred)
    return (y_pred - y_true).mean().item()


def corr(y_true, y_pred):
    y_true, y_pred = _finite_pair(y_true, y_pred)
    if y_true.numel() < 2:
        return float('nan')
    y_true = y_true - y_true.mean()
    y_pred = y_pred - y_pred.mean()
    denom = torch.sqrt((y_true ** 2).sum() * (y_pred ** 2).sum())
    return ((y_true * y_pred).sum() / denom).item() if denom > 0 else float('nan')


def r2(y_true, y_pred):
    y_true, y_pred = _finite_pair(y_true, y_pred)
    ss_res = ((y_true - y_pred) ** 2).sum()
    ss_tot = ((y_true - y_true.mean()) ** 2).sum()
    return (1 - ss_res / ss_tot).item() if ss_tot > 0 else float('nan')


def nse(y_true, y_pred):
    return r2(y_true, y_pred)


def kge(y_true, y_pred):
    y_true, y_pred = _finite_pair(y_true, y_pred)
    r = corr(y_true, y_pred)
    mean_true = y_true.mean().item()
    mean_pred = y_pred.mean().item()
    std_true = y_true.std(unbiased=False).item()
    std_pred = y_pred.std(unbiased=False).item()
    alpha = std_pred / std_true if std_true else float('nan')
    beta = mean_pred / mean_true if mean_true else float('nan')
    return 1.0 - math.sqrt((r - 1.0) ** 2 + (alpha - 1.0) ** 2 + (beta - 1.0) ** 2)


def precip_event_metrics(y_true, y_pred, threshold=EVENT_THRESHOLD):
    y_true, y_pred = _finite_pair(y_true, y_pred)
    truth = y_true >= threshold
    pred = y_pred >= threshold
    tp = (truth & pred).sum().item()
    fp = ((~truth) & pred).sum().item()
    fn = (truth & (~pred)).sum().item()
    pod = tp / (tp + fn) if (tp + fn) else float('nan')
    far = fp / (tp + fp) if (tp + fp) else float('nan')
    csi = tp / (tp + fp + fn) if (tp + fp + fn) else float('nan')
    fbias = (tp + fp) / (tp + fn) if (tp + fn) else float('nan')
    return {'pod': pod, 'far': far, 'csi': csi, 'frequency_bias': fbias}


def metrics_for_channel(y_true, y_pred, is_precip=False):
    out = {
        'rmse': rmse(y_true, y_pred),
        'mae': mae(y_true, y_pred),
        'bias': bias(y_true, y_pred),
        'corr': corr(y_true, y_pred),
        'r2': r2(y_true, y_pred),
        'nse': nse(y_true, y_pred),
        'kge': kge(y_true, y_pred),
    }
    if is_precip:
        out.update(precip_event_metrics(y_true, y_pred))
    return out


def summarize_all(y_true, y_pred):
    rows = []
    for idx, name in enumerate(TARGET_NAMES):
        row = metrics_for_channel(y_true[..., idx], y_pred[..., idx], is_precip=(name == 'precipitation'))
        row['target'] = name
        rows.append(row)
    return pd.DataFrame(rows).set_index('target')


## Model Loaders

This notebook compares four models:
- interpolation baseline
- nodewise MLP baseline
- XGBoost baseline
- GNN checkpoint

In [ ]:
def predict_interpolation(batch):
    idx = CONFIG['baselines']['interpolation']['coarse_feature_indices']
    return torch.stack([batch['x'][..., i] for i in idx], dim=-1)


def artifact_path(*parts):
    return ROOT.joinpath(*parts)


def require_artifact(path: Path, model_name: str):
    if not path.exists():
        raise FileNotFoundError(f'Missing artifact for {model_name}: {path}')
    return path


def load_mlp():
    model = build_mlp_baseline_from_config(CONFIG['baselines']['mlp'])
    state_path = require_artifact(artifact_path(CONFIG['baselines']['output_dir'], 'mlp', 'model.pt'), 'mlp')
    state = torch.load(state_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    return model


def load_xgboost():
    model_path = require_artifact(artifact_path(CONFIG['baselines']['output_dir'], 'xgboost', 'model.pkl'), 'xgboost')
    with model_path.open('rb') as handle:
        return pickle.load(handle)


def load_gnn():
    model = build_pignn_from_config(CONFIG['model'])
    checkpoint_path = require_artifact(artifact_path(CONFIG['training']['checkpoint_dir'], 'best.pt'), 'gnn')
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model_state'])
    model.to(DEVICE)
    model.eval()
    return model


In [ ]:
def collect_predictions(model_name):
    preds = []
    ys = []
    if model_name == 'interpolation':
        for batch in loader:
            preds.append(predict_interpolation(batch))
            ys.append(batch['y'])
        return torch.cat(preds, dim=0), torch.cat(ys, dim=0)

    if model_name == 'mlp':
        model = load_mlp()
        with torch.no_grad():
            for batch in loader:
                preds.append(model(batch['x'].to(DEVICE)).cpu())
                ys.append(batch['y'])
        return torch.cat(preds, dim=0), torch.cat(ys, dim=0)

    if model_name == 'xgboost':
        model = load_xgboost()
        for batch in loader:
            x = batch['x'].reshape(-1, batch['x'].shape[-1]).numpy()
            y = batch['y']
            pred = torch.from_numpy(model.predict(x)).reshape_as(y)
            preds.append(pred)
            ys.append(y)
        return torch.cat(preds, dim=0), torch.cat(ys, dim=0)

    if model_name == 'gnn':
        model = load_gnn()
        with torch.no_grad():
            for batch in loader:
                pred = model(batch['x'].to(DEVICE), batch['edge_index'].to(DEVICE), batch['edge_attr'].to(DEVICE)).cpu()
                preds.append(pred)
                ys.append(batch['y'])
        return torch.cat(preds, dim=0), torch.cat(ys, dim=0)

    raise ValueError(model_name)


## Compute Metrics

This cell loads each model artifact, generates predictions, and builds one table per model plus a combined summary.

In [ ]:
requested_models = ['interpolation', 'mlp', 'xgboost', 'gnn']
available_models = []
predictions = {}
metric_tables = {}
summary_rows = []
reference_target = None
skipped_models = {}

for model_name in requested_models:
    try:
        pred, y_true = collect_predictions(model_name)
    except FileNotFoundError as exc:
        skipped_models[model_name] = str(exc)
        print(f'skip {model_name}: {exc}')
        continue

    available_models.append(model_name)
    predictions[model_name] = pred
    if reference_target is None:
        reference_target = y_true
    table = summarize_all(y_true, pred)
    metric_tables[model_name] = table
    display(table)
    summary_rows.append({
        'model': model_name,
        'mean_rmse': table['rmse'].mean(),
        'mean_mae': table['mae'].mean(),
        'mean_kge': table['kge'].mean(),
        'precip_csi': table.loc['precipitation', 'csi'],
    })

assert summary_rows, 'No model artifacts available for comparison.'
summary = pd.DataFrame(summary_rows).sort_values('mean_rmse')
if skipped_models:
    print('skipped_models', json.dumps(skipped_models, indent=2))
summary


## Plots

The notebook makes the usual compact comparisons:
- mean metric bars across models
- per-target RMSE bars
- predicted vs target scatter for each model
- short time-series view for one node and one month
- precipitation residual histogram

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
summary.plot.bar(x='model', y='mean_rmse', ax=axes[0], legend=False, title='Mean RMSE')
summary.plot.bar(x='model', y='mean_mae', ax=axes[1], legend=False, title='Mean MAE')
summary.plot.bar(x='model', y='mean_kge', ax=axes[2], legend=False, title='Mean KGE')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()


In [ ]:
rmse_rows = []
for model_name, table in metric_tables.items():
    for target, row in table.iterrows():
        rmse_rows.append({'model': model_name, 'target': target, 'rmse': row['rmse']})
rmse_df = pd.DataFrame(rmse_rows)
pivot = rmse_df.pivot(index='target', columns='model', values='rmse')
pivot.plot.bar(figsize=(10, 5), grid=True, title='RMSE by target and model')
plt.tight_layout()


In [ ]:
model_names = available_models
sample_step = max(1, len(dataset) // 200)
flat_true = reference_target[::sample_step].reshape(-1, reference_target.shape[-1])
fig, axes = plt.subplots(len(TARGET_NAMES), len(model_names), figsize=(4 * len(model_names), 3 * len(TARGET_NAMES)), squeeze=False)
for col, model_name in enumerate(model_names):
    flat_pred = predictions[model_name][::sample_step].reshape(-1, predictions[model_name].shape[-1])
    for row, target_name in enumerate(TARGET_NAMES):
        ax = axes[row, col]
        ax.scatter(flat_true[:, row].numpy(), flat_pred[:, row].numpy(), s=4, alpha=0.2)
        ax.set_title(f'{model_name} | {target_name}')
        ax.set_xlabel('target')
        ax.set_ylabel('pred')
        ax.grid(alpha=0.2)
plt.tight_layout()


In [ ]:
node_idx = 0
time_steps = min(168, len(dataset))
fig, axes = plt.subplots(len(TARGET_NAMES), 1, figsize=(14, 10), sharex=True)
for row, target_name in enumerate(TARGET_NAMES):
    axes[row].plot(reference_target[:time_steps, node_idx, row].numpy(), label='target', linewidth=2)
    for model_name in model_names:
        axes[row].plot(predictions[model_name][:time_steps, node_idx, row].numpy(), label=model_name, alpha=0.8)
    axes[row].set_title(target_name)
    axes[row].grid(alpha=0.3)
axes[0].legend(ncol=max(1, len(model_names) + 1), fontsize=8)
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, len(model_names), figsize=(4 * len(model_names), 3), sharey=True, squeeze=False)
for ax, model_name in zip(axes[0], model_names):
    residual = (predictions[model_name][..., 1] - reference_target[..., 1]).reshape(-1).numpy()
    ax.hist(residual, bins=60, alpha=0.8)
    ax.set_title(f'{model_name} precip residual')
    ax.grid(alpha=0.2)
plt.tight_layout()


## Save Report Artifacts

This writes the summary and per-model tables so the notebook results survive kernel restarts.

In [ ]:
report_dir = ROOT / 'notebooks' / 'artifacts'
report_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(report_dir / 'comparison_summary.csv', index=False)
for model_name, table in metric_tables.items():
    table.to_csv(report_dir / f'{model_name}_metrics.csv')
with (report_dir / 'comparison_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(summary.to_dict(orient='records'), handle, indent=2)
print(report_dir)
